<br>

# FDBS

_Script_ para obter dados usando LXML.

Quero simplificar!

<br>

Michel Metran\
Data: 02.11.2025\
Atualizado em: 21.09.2026


In [ ]:
import time
from pathlib import Path

import pandas as pd

import pyFDBS
from pyFDBS import FBDS
from pyFDBS.requests.logger import FBDSLogger

In [ ]:
import os

from pyFDBS.requests.download import download_files_parallel

<br>

---

## Pastas


In [ ]:
project_path = Path('.').absolute().parents[2]
print(project_path)

# Diretório de saída
data_path = project_path / 'data'

logs_path = data_path / 'log'
logs_path.mkdir(parents=True, exist_ok=True)

output_path = data_path / 'output'
output_path.mkdir(parents=True, exist_ok=True)
output_path

<br>

---

## FBDS

Instancia a classe


In [ ]:
fdbs = FBDS(temp_path=output_path)

<br>

---

### States

Lista estados


In [ ]:
states = fdbs.get_states()
states[0:3]

In [ ]:
fdbs.states

In [ ]:
pd.DataFrame(states).head()

Obtem um estado específico


In [ ]:
state = fdbs.get_state(uf='SP')
state

<br>

---

### Municipios

Lista municípios de um estado.


In [ ]:
municipios = fdbs.get_municipalities(uf='SP')
municipios[0:3]

In [ ]:
df = pd.DataFrame(municipios)

df.info()
df.head()

In [ ]:
fdbs.municipios(uf='SP')

Obtem um município específico.


In [1]:
municipio = fdbs.get_municipalitie(
    uf="SP",
    municipality="SANTOS",
)
municipio

NameError: name 'fdbs' is not defined

<br>

---

### Layers

Lista _layers_ de um município.


In [ ]:
layers = fdbs.get_layers(
    municipality="ADAMANTINA",
    uf="SP",
)
layers[:3]

Obtem layers de um município específico.


In [ ]:
layer = fdbs.get_layer(
    municipality="ADAMANTINA",
    uf="SP",
    layer="APP",
)
layer

In [ ]:
layer = fdbs.get_layer(
    municipality="ADAMANTINA",
    uf="SP",
    layer="HIDROGRAFIA",
)
layer

<br>

---

## Logs


In [ ]:
logger = FBDSLogger(
    log_dir=logs_path,
    new_session=True,
)

<br>

---

## Download


In [ ]:
# Lista de arquivos para download (do exemplo)
files_to_download = fdbs.get_links(
    url=layer["url"],
    ignore_first=2,
)
files_to_download[0:3]

<br>

---

### ThreadPoolExecutor

Download usando threads (melhor para downloads)


In [ ]:
# 1. Marca o tempo inicial
inicio = time.perf_counter()

# Função
results = pyFDBS.download_files_parallel(
    url_list=files_to_download,
    output_dir=output_path,
    # Número de downloads simultâneos
    max_concurrent=4,
)

# 2. Marca o tempo final
fim = time.perf_counter()

# 3. Calcula a diferença
tempo_total = fim - inicio

print(f"Tempo de execução: {tempo_total:.4f} segundos")

<br>

---

### Async

Iniciando _download_ assíncrono...


In [ ]:
# 1. Marca o tempo inicial
inicio = time.perf_counter()

# Função
results = await pyFDBS.download_files_async(
    url_list=files_to_download,
    output_dir=output_path,
    max_concurrent=5,  # Número de downloads simultâneos
)

# 2. Marca o tempo final
fim = time.perf_counter()

# 3. Calcula a diferença
tempo_total = fim - inicio

print(f"Tempo de execução: {tempo_total:.4f} segundos")

<br>

---

### Resultados


In [ ]:
# Mostra resultados
print("Resultados:")
successes = [r for r in results if r["status"] == "sucesso"]
errors = [r for r in results if r["status"] == "erro"]
cached = [r for r in results if r.get("cached", False)]

print(f"Downloads com sucesso: {len(successes)} de {len(results)}")
print(f"Erros: {len(errors)} de {len(results)}")
print(f"Arquivos do cache: {len(cached)} de {len(results)}")

# Mostra erros se houver
if errors:
    print("Erros encontrados:")
    for error in errors:
        print(f"- {error['nome']}: {error['erro']}")